In [1]:
from datetime import datetime
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import TensorDataset, DataLoader
from transformers import (
    BertForSequenceClassification, 
    BertTokenizer, 
    get_linear_schedule_with_warmup
)

# --- Preprocessing Function for Sentiment Analysis with the IMDb Dataset ---

def preprocess_dataset(path):
    """ Remove unnecessary characters and encode the sentiment labels.

    The type of preprocessing required changes based on the dataset. For the
    IMDb dataset, the review texts contains HTML break tags (<br/>) leftover
    from the scraping process, and some unnecessary whitespace, which are
    removed. Finally, encode the sentiment labels as 0 for "negative" and 1 for
    "positive". This method assumes the dataset file contains the headers
    "review" and "sentiment".

    Parameters:
        path (str): A path to a dataset file containing the sentiment analysis
            dataset. The structure of the file should be as follows: one column
            called "review" containing the review text, and one column called
            "sentiment" containing the ground truth label. The label options
            should be "negative" and "positive".

    Returns:
        df_dataset (pd.DataFrame): A DataFrame containing the raw data
            loaded from the self.dataset path. In addition to the expected
            "review" and "sentiment" columns, are:

            > review_cleaned - a copy of the "review" column with the HTML
                break tags and unnecessary whitespace removed

            > sentiment_encoded - a copy of the "sentiment" column with the
                "negative" values mapped to 0 and "positive" values mapped
                to 1
    """
    df_dataset = pd.read_csv(path)

    df_dataset['review_cleaned'] = df_dataset['review'].apply(lambda x: x.replace('<br />', ' '))

    # FIX: Corrected regex for whitespace from 's+' to r'\s+'
    df_dataset['review_cleaned'] = df_dataset['review_cleaned'].replace(r'\s+', ' ', regex=True)

    df_dataset['sentiment_encoded'] = df_dataset['sentiment'].apply(lambda x: 0 if x == 'negative' else 1)

    return df_dataset

# --- Task-Agnostic Fine-tuning Pipeline Class ---

class FineTuningPipeline:

    def __init__(
            self,
            dataset,
            tokenizer,
            model,
            optimizer,
            loss_function = nn.CrossEntropyLoss(),
            val_size = 0.1,
            epochs = 4,
            seed = 42):

        self.df_dataset = dataset
        self.tokenizer = tokenizer
        self.model = model
        self.optimizer = optimizer
        self.loss_function = loss_function
        self.val_size = val_size
        self.epochs = epochs
        self.seed = seed

        # Check if GPU is available for faster training time
        if torch.cuda.is_available():
            self.device = torch.device('cuda:0')
        else:
            self.device = torch.device('cpu')

        # Perform fine-tuning
        self.model.to(self.device)
        self.set_seeds()
        self.token_ids, self.attention_masks = self.tokenize_dataset()
        self.train_dataloader, self.val_dataloader = self.create_dataloaders()
        self.scheduler = self.create_scheduler()
        self.fine_tune()

    def tokenize(self, text):
        """ Tokenize input text and return the token IDs and attention mask.

        Tokenize an input string, setting a maximum length of 512 tokens.
        Sequences with more than 512 tokens will be truncated to this limit,
        and sequences with less than 512 tokens will be supplemented with [PAD]
        tokens to bring them up to this limit. The datatype of the returned
        tensors will be the PyTorch tensor format. These return values are
        tensors of size 1 x max_length where max_length is the maximum number
        of tokens per input sequence (512 for BERT).

        Parameters:
            text (str): The text to be tokenized.

        Returns:
            token_ids (torch.Tensor): A tensor of token IDs for each token in
                the input sequence.

            attention_mask (torch.Tensor): A tensor of 1s and 0s where a 1
                indicates a token can be attended to during the attention
                process, and a 0 indicates a token should be ignored. This is
                used to prevent BERT from attending to [PAD] tokens during its
                training/inference.
        """
        # FIX: Using the tokenizer as a function call instead of .encode_plus
        # to prevent the AttributeError while maintaining the same functionality.
        batch_encoder = self.tokenizer(
            text,
            max_length = 512,
            padding = 'max_length',
            truncation = True,
            return_tensors = 'pt')

        token_ids = batch_encoder['input_ids']
        attention_mask = batch_encoder['attention_mask']

        return token_ids, attention_mask

    def tokenize_dataset(self):
        """ Apply the self.tokenize method to the fine-tuning dataset.

        Tokenize and return the input sequence for each row in the fine-tuning
        dataset given by self.dataset. The return values are tensors of size
        len_dataset x max_length where len_dataset is the number of rows in the
        fine-tuning dataset and max_length is the maximum number of tokens per
        input sequence (512 for BERT).

        Parameters:
            None.

        Returns:
            token_ids (torch.Tensor): A tensor of tensors containing token IDs
            for each token in the input sequence.

            attention_masks (torch.Tensor): A tensor of tensors containing the
                attention masks for each sequence in the fine-tuning dataset.
        """
        token_ids = []
        attention_masks = []

        for review in self.df_dataset['review_cleaned']:
            tokens, masks = self.tokenize(review)
            token_ids.append(tokens)
            attention_masks.append(masks)

        token_ids = torch.cat(token_ids, dim=0)
        attention_masks = torch.cat(attention_masks, dim=0)

        return token_ids, attention_masks

    def create_dataloaders(self):
        """ Create dataloaders for the train and validation set.

        Split the tokenized dataset into train and validation sets according to
        the self.val_size value. For example, if self.val_size is set to 0.1,
        90% of the data will be used to form the train set, and 10% for the
        validation set. Convert the "sentiment_encoded" column (labels for each
        row) to PyTorch tensors to be used in the dataloaders.

        Parameters:
            None.

        Returns:
            train_dataloader (torch.utils.data.dataloader.DataLoader): A
                dataloader of the train data, including the token IDs,
                attention masks, and sentiment labels.

            val_dataloader (torch.utils.data.dataloader.DataLoader): A
                dataloader of the validation data, including the token IDs,
                attention masks, and sentiment labels.
        """
        train_ids, val_ids = train_test_split(
                        self.token_ids,
                        test_size=self.val_size,
                        shuffle=False)

        train_masks, val_masks = train_test_split(
                                    self.attention_masks,
                                    test_size=self.val_size,
                                    shuffle=False)

        labels = torch.tensor(self.df_dataset['sentiment_encoded'].values)
        train_labels, val_labels = train_test_split(
                                        labels,
                                        test_size=self.val_size,
                                        shuffle=False)

        train_data = TensorDataset(train_ids, train_masks, train_labels)
        train_dataloader = DataLoader(train_data, shuffle=True, batch_size=16)
        val_data = TensorDataset(val_ids, val_masks, val_labels)
        val_dataloader = DataLoader(val_data, batch_size=16)

        return train_dataloader, val_dataloader

    def create_scheduler(self):
        """ Create a linear scheduler for the learning rate.

        Create a scheduler with a learning rate that increases linearly from 0
        to a maximum value (called the warmup period), then decreases linearly
        to 0 again. num_warmup_steps is set to 0 here based on an example from
        Hugging Face:
        https://github.com/huggingface/transformers/blob/5bfcd0485ece086ebcbed2
        d008813037968a9e58/examples/run_glue.py#L308
        """
        num_training_steps = self.epochs * len(self.train_dataloader)
        scheduler = get_linear_schedule_with_warmup(
            self.optimizer,
            num_warmup_steps=0,
            num_training_steps=num_training_steps)

        return scheduler

    def set_seeds(self):
        """ Set the random seeds so that results are reproduceable. """
        np.random.seed(self.seed)
        torch.manual_seed(self.seed)
        torch.cuda.manual_seed_all(self.seed)

    def fine_tune(self):
        """Train the classification head on the BERT model.

        Fine-tune the model by training the classification head (linear layer)
        sitting on top of the BERT model. The model trained on the data in the
        self.train_dataloader, and validated at the end of each epoch on the
        data in the self.val_dataloader. The series of steps are described
        below:

        Training:
        > Create a dictionary to store the average training loss and average
          validation loss for each epoch.
        > Store the time at the start of training, this is used to calculate
          the time taken for the entire training process.
        > Begin a loop to train the model for each epoch in self.epochs.

        For each epoch:
        > Switch the model to train mode. This will cause the model to behave
          differently than when in evaluation mode (e.g. the batchnorm and
          dropout layers are activated in train mode, but disabled in
          evaluation mode).
        > Set the training loss to 0 for the start of the epoch. This is used
          to track the loss of the model on the training data over subsequent
          epochs. The loss should decrease with each epoch if training is
          successful.
        > Store the time at the start of the epoch, this is used to calculate
          the time taken for the epoch to be completed.
        > As per the BERT authors' recommendations, the training data for each
          epoch is split into batches. Loop through the training process for
          each batch.

        For each batch:
        > Move the token IDs, attention masks, and labels to the GPU if
          available for faster processing, otherwise these will be kept on the
          CPU.
        > Invoke the zero_grad method to reset the calculated gradients from
          the previous iteration of this loop.
        > Pass the batch to the model to calculate the logits (predictions
          based on the current classifier weights and biases) as well as the
          loss.
        > Increment the total loss for the epoch. The loss is returned from the
          model as a PyTorch tensor so extract the float value using the item
          method.
        > Perform a backward pass of the model and propagate the loss through
          the classifier head. This will allow the model to determine what
          adjustments to make to the weights and biases to improve its
          performance on the batch.
        > Clip the gradients to be no larger than 1.0 so the model does not
          suffer from the exploding gradients problem.
        > Call the optimizer to take a step in the direction of the error
          surface as determined by the backward pass.

        After training on each batch:
        > Calculate the average loss and time taken for training on the epoch.

        Validation step for the epoch:
        > Switch the model to evaluation mode.
        > Set the validation loss to 0. This is used to track the loss of the
          model on the validation data over subsequent epochs. The loss should
          decrease with each epoch if training was successful.
        > Store the time at the start of the validation, this is used to
          calculate the time taken for the validation for this epoch to be
          completed.
        > Split the validation data into batches.

        For each batch:
        > Move the token IDs, attention masks, and labels to the GPU if
          available for faster processing, otherwise these will be kept on the
          CPU.
        > Invoke the no_grad method to instruct the model not to calculate the
          gradients since we wil not be performing any optimization steps here,
          only inference.
        > Pass the batch to the model to calculate the logits (predictions
          based on the current classifier weights and biases) as well as the
          loss.
        > Extract the logits and labels from the model and move them to the CPU
          (if they are not already there).
        > Increment the loss and calculate the accuracy based on the true
          labels in the validation dataloader.
        > Calculate the average loss and accuracy, and add these to the loss
          dictionary.
        """

        loss_dict = {
            'epoch': [i+1 for i in range(self.epochs)],
            'average training loss': [],
            'average validation loss': []
        }

        t0_train = datetime.now()

        for epoch in range(0, self.epochs):

            # Train step
            self.model.train()
            training_loss = 0
            t0_epoch = datetime.now()

            print(f'{"-"*20} Epoch {epoch+1} {"-"*20}')
            print('Training:\n---------')
            print(f'Start Time:       {t0_epoch}')

            for batch in self.train_dataloader:

                batch_token_ids = batch[0].to(self.device)
                batch_attention_mask = batch[1].to(self.device)
                batch_labels = batch[2].to(self.device)

                self.model.zero_grad()

                # Optimized for modern transformers library (returning dict)
                outputs = self.model(
                    batch_token_ids,
                    token_type_ids = None,
                    attention_mask=batch_attention_mask,
                    labels=batch_labels)

                loss = outputs.loss
                training_loss += loss.item()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                self.optimizer.step()
                self.scheduler.step()

            average_train_loss = training_loss / len(self.train_dataloader)
            time_epoch = datetime.now() - t0_epoch

            print(f'Average Loss:     {average_train_loss}')
            print(f'Time Taken:       {time_epoch}')

            # Validation step
            self.model.eval()
            val_loss = 0
            val_accuracy = 0
            t0_val = datetime.now()

            print('Validation:\n---------')
            print(f'Start Time:       {t0_val}')

            for batch in self.val_dataloader:

                batch_token_ids = batch[0].to(self.device)
                batch_attention_mask = batch[1].to(self.device)
                batch_labels = batch[2].to(self.device)

                with torch.no_grad():
                    outputs = self.model(
                        batch_token_ids,
                        attention_mask = batch_attention_mask,
                        labels = batch_labels,
                        token_type_ids = None)

                logits = outputs.logits.detach().cpu().numpy()
                label_ids = batch_labels.to('cpu').numpy()
                val_loss += outputs.loss.item()
                val_accuracy += self.calculate_accuracy(logits, label_ids)

            average_val_accuracy = val_accuracy / len(self.val_dataloader)
            average_val_loss = val_loss / len(self.val_dataloader)
            time_val = datetime.now() - t0_val

            print(f'Average Loss:     {average_val_loss}')
            print(f'Average Accuracy: {average_val_accuracy}')
            print(f'Time Taken:       {time_val}\n')

            loss_dict['average training loss'].append(average_train_loss)
            loss_dict['average validation loss'].append(average_val_loss)

        print(f'Total training time: {datetime.now()-t0_train}')

    def calculate_accuracy(self, preds, labels):
        """ Calculate the accuracy of model predictions against true labels.

        Parameters:
            preds (np.array): The predicted label from the model
            labels (np.array): The true label

        Returns:
            accuracy (float): The accuracy as a percentage of the correct
                predictions.
        """
        pred_flat = np.argmax(preds, axis=1).flatten()
        labels_flat = labels.flatten()
        accuracy = np.sum(pred_flat == labels_flat) / len(labels_flat)

        return accuracy

    def predict(self, dataloader):
        """Return the predicted probabilities of each class for input text.

        Parameters:
            dataloader (torch.utils.data.DataLoader): A DataLoader containing
                the token IDs and attention masks for the text to perform
                inference on.

        Returns:
            probs (np.array): A numpy array containing the probability values
                for each class as predicted by the model.
        """

        self.model.eval()
        all_logits = []

        for batch in dataloader:

            batch_token_ids, batch_attention_mask = tuple(t.to(self.device) 
                for t in batch)[:2]

            with torch.no_grad():
                outputs = self.model(batch_token_ids, attention_mask=batch_attention_mask)

            all_logits.append(outputs.logits)

        all_logits = torch.cat(all_logits, dim=0)

        probs = F.softmax(all_logits, dim=1).cpu().numpy()
        return probs

# --- Example of Using the Class for Sentiment Analysis with the IMDb Dataset ---

# 1. Update path for Kaggle
path = '/kaggle/input/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv' 
dataset = preprocess_dataset(path)

# 2. Slice for performance on Kaggle (e.g., first 1000 rows)
dataset = dataset.iloc[:1000].reset_index(drop=True)

# 3. Load BERT components
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2)

# 4. Standard optimizer setup
optimizer = AdamW(model.parameters(), lr=2e-5)

# 5. Initialize pipeline and start fine-tuning
fine_tuned_pipeline = FineTuningPipeline(
    dataset = dataset,
    tokenizer = tokenizer,
    model = model,
    optimizer = optimizer,
    epochs = 2,
    seed = 42
)

# 6. Make predictions using the validation dataloader stored in the pipeline
val_probs = fine_tuned_pipeline.predict(fine_tuned_pipeline.val_dataloader)
print(f"Sample Predictions (Probabilities):\n{val_probs[:5]}")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


-------------------- Epoch 1 --------------------
Training:
---------
Start Time:       2026-04-01 18:30:03.251836
Average Loss:     0.47677061641425417
Time Taken:       0:01:20.125340
Validation:
---------
Start Time:       2026-04-01 18:31:23.378197
Average Loss:     0.2558211726801736
Average Accuracy: 0.9017857142857143
Time Taken:       0:00:03.366481

-------------------- Epoch 2 --------------------
Training:
---------
Start Time:       2026-04-01 18:31:26.745820
Average Loss:     0.1990885967878919
Time Taken:       0:01:32.602917
Validation:
---------
Start Time:       2026-04-01 18:32:59.349713
Average Loss:     0.22707484609314374
Average Accuracy: 0.9196428571428571
Time Taken:       0:00:03.802198

Total training time: 0:02:59.901114
Sample Predictions (Probabilities):
[[0.02503294 0.97496706]
 [0.94943506 0.05056495]
 [0.93555    0.06445003]
 [0.9331123  0.06688771]
 [0.8409263  0.15907374]]


In [2]:
import numpy as np

# 1. Convert probabilities to class labels (0 or 1)
# axis=1 means "look across the rows"
predicted_labels = np.argmax(val_probs, axis=1)

# 2. Compare the first 10 predictions to the actual probabilities
print(f"Probabilities:\n{val_probs[:5]}")
print(f"Predicted Labels: {predicted_labels[:5]}")

Probabilities:
[[0.02503294 0.97496706]
 [0.94943506 0.05056495]
 [0.93555    0.06445003]
 [0.9331123  0.06688771]
 [0.8409263  0.15907374]]
Predicted Labels: [1 0 0 0 0]


In [3]:
import os

# Create a directory for the saved model
save_directory = "./saved_imdb_bert"
if not os.path.exists(save_directory):
    os.makedirs(save_directory)

# Save the model
# This saves the 'pytorch_model.bin' and 'config.json'
fine_tuned_pipeline.model.save_pretrained(save_directory)

# Save the tokenizer
# This saves 'vocab.txt', 'tokenizer_config.json', etc.
fine_tuned_pipeline.tokenizer.save_pretrained(save_directory)

print(f"Model and Tokenizer successfully saved to {save_directory}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and Tokenizer successfully saved to ./saved_imdb_bert


In [4]:
from transformers import BertForSequenceClassification, BertTokenizer
import torch

# Load from the folder we just created
loaded_tokenizer = BertTokenizer.from_pretrained("./saved_imdb_bert")
loaded_model = BertForSequenceClassification.from_pretrained("./saved_imdb_bert")

# Quick Test: Predict sentiment for a new sentence
text = "This movie was a total waste of time, I hated it."
inputs = loaded_tokenizer(text, return_tensors="pt", truncation=True, padding=True)

with torch.no_grad():
    outputs = loaded_model(**inputs)
    prediction = torch.argmax(outputs.logits, dim=1).item()

print(f"Sentiment Prediction (0=Neg, 1=Pos): {prediction}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Sentiment Prediction (0=Neg, 1=Pos): 0


In [5]:
import shutil

# This will create a file named 'my_bert_model.zip' in your /kaggle/working/ directory
shutil.make_archive('my_bert_model', 'zip', './saved_imdb_bert')

print("Zip file created! Check the 'Output' section in the right-hand sidebar to download 'my_bert_model.zip'.")

Zip file created! Check the 'Output' section in the right-hand sidebar to download 'my_bert_model.zip'.


In [6]:
def test_sentiment(text_list):
    # Prepare inputs for the model
    inputs = loaded_tokenizer(text_list, padding=True, truncation=True, return_tensors="pt")
    
    # Move to GPU if available
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    loaded_model.to(device)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = loaded_model(**inputs)
        # Convert logits to probabilities
        probs = F.softmax(outputs.logits, dim=1).cpu().numpy()
        # Get the final 0 or 1 labels
        labels = np.argmax(probs, axis=1)
        
    for text, prob, label in zip(text_list, probs, labels):
        sentiment = "POSITIVE" if label == 1 else "NEGATIVE"
        confidence = prob[label] * 100
        print(f"Review: '{text}'")
        print(f"Prediction: {sentiment} ({confidence:.2f}% confidence)\n")

# --- Try these custom sentences ---
my_reviews = [
    "I've never seen such a boring movie in my life. A complete waste of money.",
    "The plot was okay, but the acting from the lead star was absolutely incredible!",
    "It was not as good as the first one, but still worth a watch for the visuals.",
    "Avoid this like the plague. Total disaster."
]

test_sentiment(my_reviews)

Review: 'I've never seen such a boring movie in my life. A complete waste of money.'
Prediction: NEGATIVE (94.06% confidence)

Review: 'The plot was okay, but the acting from the lead star was absolutely incredible!'
Prediction: POSITIVE (91.49% confidence)

Review: 'It was not as good as the first one, but still worth a watch for the visuals.'
Prediction: POSITIVE (69.60% confidence)

Review: 'Avoid this like the plague. Total disaster.'
Prediction: NEGATIVE (83.06% confidence)

